# Module 03 - Chunking and Ingestion

**Duration:** 45 minutes

Chunking is how you split documents into pieces before embedding them.
It has more impact on retrieval quality than almost any other decision in the pipeline.
This module covers the main strategies, explores the chunker built into this repo,
and walks through ingesting a set of documents into ChromaDB.

---


## 3.1 Why chunking matters

Embedding models have a token limit (typically 256 to 512 tokens).
A long document cannot be embedded as a single vector — it has to be split.

But even ignoring the token limit, chunking matters for retrieval quality:

- A chunk that is **too short** may lack enough context to match the query correctly.
- A chunk that is **too long** contains too many topics; its embedding becomes a blurry
  average that matches many things weakly rather than one thing strongly.
- **Splitting mid-sentence** or mid-paragraph destroys the meaning of the pieces.

There is no universally correct chunk size. It depends on the documents,
the embedding model, and the kinds of questions being asked.
The goal is to produce chunks that are **self-contained and focused**.

### The retrieval granularity trade-off

Chunking is fundamentally a precision vs. recall trade-off:

| Chunk size | Retrieval behaviour | Risk |
|------------|--------------------|----|
| Very small (< 50 words) | Highly specific matches | Loses surrounding context; hard to answer multi-sentence questions |
| Medium (100–300 words) | Good balance | Occasional topic bleed at boundaries |
| Large (400+ words) | Broad coverage per chunk | Embedding is diluted; many irrelevant sentences retrieved |

A common practical choice is **150–250 words** with a **10–20% overlap**,
but this should be validated against your specific documents and queries.

### Chunking strategy overview

There are four main families of chunking strategies:

1. **Fixed-size** — split every N words. Simple but crude.
2. **Overlap** — like fixed-size, but repeat a window of text at each boundary.
3. **Structure-aware** — split on sentences, paragraphs, or section headings.
4. **Semantic** — use embeddings to detect topic shifts and split there.

We will implement 1, 2, and 3 in this module. Strategy 4 is more complex
but covered briefly in the exercises.


In [ ]:
# Read a sample document so we have something to work with
import sys
sys.path.insert(0, '..')  # only needed if running outside uv environment

from ragsst.utils import read_file

text = read_file('../data/sample_docs/aihpi-home.txt')
print(f'Document length: {len(text.split())} words')
print('\nFirst 300 characters:')
print(text[:300])


## 3.2 Strategy 1: fixed-size chunking

Split the text into chunks of exactly N words, regardless of sentence or paragraph boundaries.
Simple to implement, but the chunks often cut mid-sentence.


In [ ]:
from ragsst.utils import split_text_basic

chunks_basic = split_text_basic(text, max_words=100)

print(f'Number of chunks: {len(chunks_basic)}')
print('\nFirst 3 chunks:')
for i, c in enumerate(chunks_basic[:3]):
    print(f'\n--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c)


## 3.3 Strategy 2: structure-aware chunking

The `split_text` function in this repo is smarter: it treats short lines
(likely headings or titles) as section boundaries and keeps them attached
to the paragraph that follows. This preserves section context that
fixed-size chunking would throw away.

### Why headings matter

Consider a document with this structure:

```
Refund Policy
You may return any item within 30 days for a full refund...

Shipping Policy
Standard shipping takes 3–5 business days...
```

If you split at a word boundary in the middle of "Refund Policy / You may return...",
you get a chunk that starts with "...30 days for a full refund" — which, without
the heading, looks like it could be about almost anything.

The structure-aware chunker keeps "Refund Policy" attached to its paragraph,
so the chunk reads: *"Refund Policy: You may return any item within 30 days..."*
That chunk will now embed close to queries about refunds, returns, and policies.


In [ ]:
from ragsst.utils import split_text

chunks_smart = split_text(text, max_words=100)

print(f'Number of chunks: {len(chunks_smart)}')
print('\nFirst 3 chunks:')
for i, c in enumerate(chunks_smart[:3]):
    print(f'\n--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c)


Notice how the smart chunker keeps section headings attached to their content.
This is the kind of contextual information that helps the embedder produce
a more accurate vector for the chunk.


In [ ]:
# Compare chunk counts and average chunk size for different max_words settings
print(f"{'max_words':>12}  {'chunks':>8}  {'avg words':>10}")
print('-' * 35)
for max_w in [64, 128, 256, 512]:
    chunks = split_text(text, max_words=max_w)
    avg = sum(len(c.split()) for c in chunks) / len(chunks)
    print(f'{max_w:>12}  {len(chunks):>8}  {avg:>10.1f}')


**Exercise:** Try max_words=50 and max_words=400. Look at the resulting chunks.
At what size do the chunks stop being self-contained?


## 3.4 Overlap

One common problem with any chunking strategy is that relevant information
can end up split across a chunk boundary. A sentence that starts at the
end of chunk 3 and finishes at the start of chunk 4 will be poorly represented
in both chunks.

**Overlapping chunks** solve this by repeating a small window of text at each boundary.
A typical overlap is 10–20% of the chunk size (e.g. 20 words for a 100-word chunk).

The trade-off is that you store more data and may retrieve near-duplicate content.
The duplicate issue can be handled downstream (Module 05 shows deduplication in multi-query).

### Overlap vs. parent-child chunking

An alternative to simple overlap is **parent-child chunking**:
- Embed small child chunks for high-precision retrieval
- When a child chunk is retrieved, return its larger parent chunk to the LLM

This gives you the best of both worlds: precise matching AND full context.
LlamaIndex calls this "small-to-big retrieval". It is more complex to implement
but meaningfully improves answer quality for long documents.


In [ ]:
def split_with_overlap(text: str, max_words: int = 200, overlap: int = 30) -> list[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(' '.join(words[start:end]))
        start += max_words - overlap
    return chunks


chunks_overlap = split_with_overlap(text, max_words=100, overlap=20)
print(f'Chunks with overlap: {len(chunks_overlap)}')

# Show the boundary between chunk 0 and chunk 1
print('\nEnd of chunk 0:')
print(' '.join(chunks_overlap[0].split()[-15:]))
print('\nStart of chunk 1:')
print(' '.join(chunks_overlap[1].split()[:15]))


## 3.4b Document loading: what happens before chunking

Before chunking, documents must be read into plain text.
This step is called **document loading** and hides surprising complexity:

| Format | Challenge |
|--------|-----------|
| `.txt` | Easy — read directly |
| `.pdf` | May have text layer (digital) or only images (scanned) |
| `.docx` | Paragraphs, tables, headers need structure extraction |
| HTML | Must strip navigation, ads, boilerplate |
| Tables | Cell boundaries are meaningful; flattening loses structure |

The `read_file` function in `ragsst.utils` handles `.txt`, `.pdf`, and `.docx`.
For more formats, LangChain's `DocumentLoader` ecosystem covers hundreds of sources
(Notion, Confluence, email, spreadsheets, etc.) — we cover this in Bonus A.

### Metadata: what to store alongside chunks

Each chunk should be stored with metadata that helps with:
1. **Source attribution** — which document did this come from?
2. **Filtering** — only search documents from a certain date or category
3. **Deduplication** — avoid returning two chunks from the same source

In this repo, the `make_collection` method stores `{'source': filename, 'part': chunk_index}`
for each chunk. You can see this in the inspection cell below.


## 3.5 Ingesting documents into ChromaDB

Now we put it together: read documents, chunk them, embed them, and store them
in a persistent ChromaDB collection.
The `RAGTool.make_collection()` method handles all of this.


In [ ]:
from ragsst.ragtool import RAGTool

tool = RAGTool(
    data_path='../data/sample_docs',
    collection_name='workshop_docs',
)

tool.make_collection('../data/sample_docs', 'workshop_docs')
print(f'\nCollection has {tool.collection.count()} chunks.')


In [ ]:
# Inspect what was ingested
sample = tool.collection.get(limit=5, include=['documents', 'metadatas'])

print('Sample chunks from the collection:\n')
for doc, meta in zip(sample['documents'], sample['metadatas']):
    print(f"Source: {meta['source']} (part {meta['part']})")
    print(doc[:150])
    print()


In [ ]:
# Quick retrieval check
result = tool.get_relevant_text('What AI services are available?', nresults=2)
print('Retrieved context:')
print(result)


---

**Exercises**

1. Open `src/ragsst/utils.py` and read `split_text`. Trace through what happens
   when a heading line (e.g. 'AI Workshops') is encountered. Why is it kept with the next paragraph?

2. Create a new text file in `data/sample_docs/` with a few paragraphs about a topic
   you know well. Run `make_collection` again and query for something from your file.

3. Call `tool.collection.get()` without a limit and look at how many chunks
   came from each source file. Does the number of chunks seem proportional to the
   length of each document?

---

**Further reading**

- Chunking strategies overview: https://www.pinecone.io/learn/chunking-strategies/
- Rethinking chunk size: https://arxiv.org/abs/2505.21700
